# Data Cleaning - Mutual Fund Analysis

## Objective

The objective of this notebook is to clean the raw datasets before analysis and dashboard creation.

The following datasets are cleaned:

1. NAV History
2. Investor Transactions
3. Scheme Performance

Cleaning includes:

- Handling missing values
- Removing duplicate records
- Standardizing data
- Validating numerical values
- Saving cleaned datasets

# 1. Cleaning NAV History Dataset

Cleaning steps:

- Load dataset
- Convert Date column to datetime
- Sort data
- Remove duplicate rows
- Forward fill missing NAV values
- Validate NAV values (>0)
- Save cleaned dataset

In [1]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent

raw_path = project_root / "data" / "raw" / "02_nav_history.csv"
processed_path = project_root / "data" / "processed" / "02_nav_history_cleaned.csv"

df = pd.read_csv(raw_path)

print("Original Shape:", df.shape)

df.head()

Original Shape: (46000, 3)


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  str    
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 1.1 MB


In [3]:
print(df.isnull().sum())

amfi_code    0
date         0
nav          0
dtype: int64


In [4]:
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(
    by=["amfi_code","date"]
)

duplicates = df.duplicated().sum()

print("Duplicate Rows:",duplicates)

df = df.drop_duplicates()

missing_before = df["nav"].isnull().sum()

df["nav"] = df.groupby("amfi_code")["nav"].ffill()

missing_after = df["nav"].isnull().sum()

print("Missing Before:",missing_before)
print("Missing After :",missing_after)

invalid = df[df["nav"]<=0]

print("Invalid NAV Records:",len(invalid))

df = df[df["nav"]>0]

Duplicate Rows: 0
Missing Before: 0
Missing After : 0
Invalid NAV Records: 0


In [5]:
df.to_csv(processed_path,index=False)

print("Saved Successfully")

print(df.shape)

df.head()

Saved Successfully
(46000, 3)


,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


## Result

The NAV History dataset has been successfully cleaned by:

- removing duplicates
- handling missing NAV values
- validating NAV values
- saving cleaned dataset

# 2. Cleaning Investor Transactions Dataset

Cleaning Steps

- Convert transaction date
- Remove duplicate rows
- Standardize transaction types
- Validate transaction amount
- Validate KYC Status
- Save cleaned dataset

In [6]:
raw_path = project_root / "data" / "raw" / "08_investor_transactions.csv"
processed_path = project_root / "data" / "processed" / "08_investor_transactions_cleaned.csv"

df = pd.read_csv(raw_path)

print("Original Shape:",df.shape)

df.head()

Original Shape: (32778, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [7]:
df["transaction_date"] = pd.to_datetime(df["transaction_date"])

duplicates = df.duplicated().sum()

print("Duplicate Rows:",duplicates)

df = df.drop_duplicates()

df["transaction_type"] = (
    df["transaction_type"]
        .str.strip()
        .str.title()
)

mapping={
    "Sip":"SIP",
    "Lumpsum":"Lumpsum",
    "Redemption":"Redemption"
}

df["transaction_type"]=df["transaction_type"].replace(mapping)

print(df["transaction_type"].unique())

invalid_amount=(df["amount_inr"]<=0).sum()

print("Invalid Amount:",invalid_amount)

df=df[df["amount_inr"]>0]

valid_kyc=["Verified","Pending"]

invalid=df[~df["kyc_status"].isin(valid_kyc)]

print("Invalid KYC:",len(invalid))

Duplicate Rows: 0
<StringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str
Invalid Amount: 0
Invalid KYC: 0


In [8]:
df.to_csv(processed_path,index=False)

print("Saved Successfully")

print(df.shape)

df.head()

Saved Successfully
(32778, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


## Result

Investor Transactions dataset cleaned successfully.

Cleaning performed:

- Duplicate removal
- Transaction type standardization
- Amount validation
- KYC validation

# 3. Cleaning Scheme Performance Dataset

Cleaning Steps

- Convert return columns to numeric
- Check missing values
- Validate expense ratio
- Detect anomalies
- Remove duplicates
- Save cleaned dataset

In [9]:
raw_path = project_root / "data" / "raw" / "07_scheme_performance.csv"
processed_path = project_root / "data" / "processed" / "07_scheme_performance_cleaned.csv"

df = pd.read_csv(raw_path)

print("Original Shape:",df.shape)

df.head()

Original Shape: (40, 19)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [10]:
return_columns=[
"return_1yr_pct",
"return_3yr_pct",
"return_5yr_pct",
"benchmark_3yr_pct",
"alpha",
"beta",
"sharpe_ratio",
"sortino_ratio",
"std_dev_ann_pct",
"max_drawdown_pct",
"expense_ratio_pct"
]

for col in return_columns:
    df[col]=pd.to_numeric(df[col],errors="coerce")

print(df[return_columns].isnull().sum())

return_1yr_pct       0
return_3yr_pct       0
return_5yr_pct       0
benchmark_3yr_pct    0
alpha                0
beta                 0
sharpe_ratio         0
sortino_ratio        0
std_dev_ann_pct      0
max_drawdown_pct     0
expense_ratio_pct    0
dtype: int64


In [11]:
invalid=df[
(df["expense_ratio_pct"]<0.1)|
(df["expense_ratio_pct"]>2.5)
]

print("Invalid Expense Ratio:",len(invalid))

anomalies=df[
(df["return_1yr_pct"]<-100)|
(df["return_1yr_pct"]>200)
]

print("Return Anomalies:",len(anomalies))

duplicates=df.duplicated().sum()

print("Duplicates:",duplicates)

df=df.drop_duplicates()

Invalid Expense Ratio: 0
Return Anomalies: 0
Duplicates: 0


In [12]:
df.to_csv(processed_path,index=False)

print("Saved Successfully")

print(df.shape)

df.head()

Saved Successfully
(40, 19)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


## Result

Scheme Performance dataset cleaned successfully.

Cleaning included:

- Numeric conversion
- Missing value identification
- Expense ratio validation
- Return anomaly detection
- Duplicate removal

# Conclusion

The following datasets were successfully cleaned and saved into the `data/processed` directory:

- 02_nav_history_cleaned.csv
- 07_scheme_performance_cleaned.csv
- 08_investor_transactions_cleaned.csv

These cleaned datasets are now ready for:

- SQL Database Loading
- Exploratory Data Analysis
- Power BI Dashboard Development
- Performance Metrics Calculation